#### NOTES DE COHERENCE (**à retirer à postériori**)
- Analyse si il y a pas de logement observé et/ou pas de carte d'électeurs vs type de ménage
- Prendre en considération si c'est le/la cheffe de ménage qui répond ou non et son statut mat / decision / status
- Prendre en considération le lieu (village)
- Vérifier que l'âge a été "constraint" - 🟠 Vérification de <span style="color: orange">niveau intermédiaire</span> : ⚠️ voir dans quels cas les mineurs sont autorisés à participer à l'enquête (possibilité d'ajout de questions représentants/accompagnées) -> *vérification technique* 
- Vérifier les questions de protection - 🔴 Vérification de <span style="color: red">niveau élevé</span> -> 

## 1. Analyse Univariée de la Table principale

In [18]:
import sqlite3
import re
import pandas as pd

# ==========================================
# 1. CONNEXION ET CHARGEMENT DES DONNÉES
# ==========================================
db_path = r"C:\Users\maril\Documents\Enquête Multisectorielle - ONG\outputs\enquete_ms.db"
conn = sqlite3.connect(db_path)

nom_table_reelle = "META_ENQUETE"
df_data = pd.read_sql(f"SELECT * FROM {nom_table_reelle}", conn)
conn.close()

chemin_xlsform = r"C:\Users\maril\Documents\Enquête Multisectorielle - ONG\data\Questionnaire Vulnérabilités Fictif.xlsx"
df_survey = pd.read_excel(chemin_xlsform, sheet_name="survey")

print(f"✅ Données chargées : {len(df_data)} lignes et {len(df_data.columns)} colonnes.")

# --- LISTES D'EXCLUSION ÉLARGIES ---
COLONNES_EXCLUES_TECHNIQUES = ['id', 'uuid', 'index', 'jeton', 'token', 'key', '_id', '_uuid']
MOTS_CLES_EXCLUS_TEMPORELS = ['start', 'end', 'today', 'deviceid', 'subscriberid', 'simserial', 'phonenumber', 'date', 'heure']


# ==========================================
# 2. PROPAGATION ROBUSTE DES GROUPES ET TRADUCTION
# ==========================================
def preparer_regles_survey(df_survey):
    lignes_traitees = []
    pile_groupes = []
    
    for _, row in df_survey.iterrows():
        col_type = str(row.get('type', ''))
        col_name = row.get('name')
        relevant_actuel = row.get('relevant')
        
        if col_type.startswith('begin_group') or col_type.startswith('begin group'):
            pile_groupes.append(relevant_actuel)
            continue
        elif col_type.startswith('end_group') or col_type.startswith('end group'):
            if pile_groupes:
                pile_groupes.pop()
            continue
            
        if pd.notna(col_name):
            tous_les_relevant = []
            for g_rel in pile_groupes:
                if pd.notna(g_rel) and str(g_rel).strip():
                    tous_les_relevant.append(str(g_rel).strip())
            
            if pd.notna(relevant_actuel) and str(relevant_actuel).strip():
                tous_les_relevant.append(str(relevant_actuel).strip())
                
            combined_relevant = " & ".join([f"({r})" for r in tous_les_relevant]) if tous_les_relevant else None
            
            row_dict = row.to_dict()
            row_dict['relevant_combine'] = combined_relevant
            lignes_traitees.append(row_dict)
            
    return pd.DataFrame(lignes_traitees)


def traduire_relevant_en_pandas(relevant_expr):
    if pd.isna(relevant_expr) or not str(relevant_expr).strip():
        return None
        
    expr_pandas = str(relevant_expr)
    
    if "selected(" in expr_pandas:
        match = re.search(r"selected\(\s*\${([^}]+)}\s*,\s*'([^']+)'\)", expr_pandas)
        if match:
            col, val = match.groups()
            expr_pandas = f"df_data['{col}'] == '{val}'"
    
    expr_pandas = re.sub(r'\$\{([^}]+)\}', r"df_data['\1']", expr_pandas)
    expr_pandas = expr_pandas.replace(' and ', ' & ').replace(' or ', ' | ')
    
    return expr_pandas


def rendre_relevant_lisible(expr_brute):
    """Transforme une expression XLSForm brute en phrase claire et compréhensible."""
    if pd.isna(expr_brute) or not str(expr_brute).strip():
        return ""
    
    texte = str(expr_brute).strip()
    
    # Nettoyage récursif des parenthèses englobantes
    while texte.startswith('(') and texte.endswith(')'):
        texte = texte[1:-1].strip()
        
    texte = re.sub(r'\$\{([^}]+)\}', r'\1', texte)
    texte = texte.replace('selected(', '').replace(')', '')
    
    if ',' in texte:
        parts = [p.strip().replace("'", "").replace('"', '') for p in texte.split(',')]
        if len(parts) == 2:
            return f"La question s'applique si la variable '{parts[0]}' vaut '{parts[1]}'"
    elif '==' in texte:
        parts = [p.strip().replace("'", "").replace('"', '') for p in texte.split('==')]
        if len(parts) == 2:
            return f"La question s'applique si la variable '{parts[0]}' vaut '{parts[1]}'"
            
    return f"Condition : {texte}"


# ==========================================
# 3. MOTEUR D'ANALYSE UNIVARIÉE INTELLIGENT
# ==========================================
def executer_eda_univariee(df_data, df_survey_clean):
    total_global = len(df_data)
    colonnes_xlsform = set(df_survey_clean['name'].dropna().tolist())
    colonnes_orphelines = list(set(df_data.columns) - colonnes_xlsform)
    
    for _, row in df_survey_clean.iterrows():
        col_name = row.get('name')
        col_type = str(row.get('type'))
        relevant_rule = row.get('relevant_combine')
        
        if pd.isna(col_name) or col_name not in df_data.columns:
            continue
            
        if any(kw in col_name.lower() for kw in COLONNES_EXCLUES_TECHNIQUES + MOTS_CLES_EXCLUS_TEMPORELS):
            continue
            
        df_cible = df_data.copy()
        filtre_pandas = traduire_relevant_en_pandas(relevant_rule)
        
        total_eligible = total_global
        masque_valide = True
        
        if filtre_pandas:
            try:
                mask = eval(filtre_pandas)
                df_cible = df_data[mask]
                total_eligible = len(df_cible)
            except Exception as e:
                masque_valide = False
                total_eligible = 0 

        print(f"\n==================================================")
        print(f"🔍 Analyse : {col_name} (Type XLSForm : {col_type})")
        print(f"--------------------------------------------------")
        
        if filtre_pandas and masque_valide:
            print(f"📌 Conditionnement : {rendre_relevant_lisible(relevant_rule)}")
            print(f"   - Éligibles : {total_eligible} / {total_global} lignes")
        elif filtre_pandas and not masque_valide:
            print(f"📌 Conditionnement : {relevant_rule}")
            print(f"   - ⚠️ Erreur d'évaluation du filtre - Vérifiez la syntaxe")
            
        if total_eligible == 0 and filtre_pandas:
            print("⚠️ Aucun enregistrement ou échec du filtre éligible.")
            continue
            
        # --- INFÉRENCE INTELLIGENTE DES TYPES ---
        if col_type == 'text':
            series_val = df_cible[col_name].dropna()
            nb_uniques = series_val.nunique()
            if 0 < nb_uniques <= 6:
                print(f"💡 [Inférence] Type 'text' converti en select_one (faible cardinalité : {nb_uniques} valeurs uniques).")
                col_type = 'select_one_inferred'

        # --- 1. STATISTIQUES NUMÉRIQUES ---
        if col_type.startswith('integer') or col_type.startswith('decimal'):
            nb_nuls = df_cible[col_name].isnull().sum()
            print(f"📊 Statistiques Numériques (Valeurs manquantes : {nb_nuls}) :")
            stats = df_cible[col_name].describe()[['mean', 'std', 'min', '50%', 'max']]
            stats.index = ['Moyenne', 'Écart-type', 'Minimum', 'Médiane', 'Maximum']
            print(stats.round(2))
            
        # --- 2. TABLEAUX DE FRÉQUENCES (CATÉGORIELS) ---
        elif col_type.startswith('select_one') or col_type.startswith('select_multiple') or col_type == 'select_one_inferred':
            freq = df_cible[col_name].value_counts(dropna=False)
            freq_pct = (freq / total_eligible) * 100
            
            resume = pd.DataFrame({
                'Effectif': freq, 
                'Pourcentage (%)': freq_pct.round(2)
            })
            
            resume.index = ['NaN (non concernés)' if pd.isna(x) else str(x) for x in resume.index]
            
            total_effectif = resume['Effectif'].sum()
            total_pct = (total_effectif / total_eligible) * 100 if total_eligible > 0 else 0
            df_total = pd.DataFrame({
                'Effectif': [total_effectif],
                'Pourcentage (%)': [round(total_pct, 2)]
            }, index=['TOTAL'])
            
            print(f"📋 Tableau de fréquences (Total catégories : {len(freq)}) :")
            if len(freq) > 20:
                print(pd.concat([resume.head(25), df_total]).to_string())
            else:
                print(pd.concat([resume, df_total]).to_string())

        # --- 3. NLP POUR LES TEXTES LIBRES LONGS ---
        elif col_type == 'text':
            series_text = df_cible[col_name].dropna().astype(str)
            series_text = series_text[series_text.str.strip() != '']
            print(f"📝 Analyse NLP / Texte libre ({len(series_text)} réponses) :")
            
            if len(series_text) > 0:
                longueurs_mots = series_text.apply(lambda x: len(x.split()))
                longueurs_car = series_text.apply(len)
                print(f"   - Longueur moyenne : {longueurs_mots.mean():.1f} mots ({longueurs_car.mean():.1f} caractères)")
                
                stopwords_fr = {'le', 'la', 'les', 'un', 'une', 'des', 'et', 'en', 'à', 'de', 'du', 'au', 'aux', 'que', 'qui', 'dans', 'sur', 'par', 'pour', 'avec', 'pas', 'plus', 'sont', 'est', 'ont', 'nous', 'vous', 'ils', 'elles', 'leur', 'leurs', 'son', 'sa', 'ses', 'ce', 'cette', 'ces', 'mais', 'ou', 'donc', 'or', 'ni', 'car', 'etre', 'avoir', 'suis', 'es', 'sommes', 'etes', 'sont', 'ai', 'as', 'avons', 'avez', 'ont', 'tout', 'tous', 'toute', 'toutes', 'faire', 'fait'}
                
                tous_les_mots = " ".join(series_text).lower()
                mots_propres = re.findall(r'\b[a-zA-Zàâäéèêëïîôöùûüç]{4,}\b', tous_les_mots)
                mots_filtres = [m for m in mots_propres if m not in stopwords_fr]
                
                top_mots = pd.Series(mots_filtres).value_counts().head(5)
                print(f"   - Top mots significatifs : {dict(top_mots)}")
                
                print(f"   - Top 10 des verbatims / commentaires les plus fréquents (extraits) :")
                top_commentaires = series_text.value_counts().head(10)
                for comm, count in top_commentaires.items():
                    print(f"     * ({count}x) \"{comm}\'")
            else:
                print("   - Aucune réponse textuelle enregistrée.")

    # --- 4. TRAITEMENT AUTOMATIQUE DES COLONNES DE NETTOYAGE (ex: duree_minutes) ---
    print(f"\n==================================================")
    print(f"🔍 ANALYSE DES COLONNES CALCULÉES / NETTOYAGE (HORS XLSFORM)")
    print(f"==================================================")
    
    for col_name in colonnes_orphelines:
        if any(kw in col_name.lower() for kw in COLONNES_EXCLUES_TECHNIQUES):
            continue
            
        if pd.api.types.is_numeric_dtype(df_data[col_name]):
            print(f"\n📊 Statistiques Numériques (Calculée) : {col_name}")
            nb_nuls = df_data[col_name].isnull().sum()
            print(f"   - Valeurs manquantes : {nb_nuls}")
            stats = df_data[col_name].describe()[['mean', 'std', 'min', '50%', 'max']]
            stats.index = ['Moyenne', 'Écart-type', 'Minimum', 'Médiane', 'Maximum']
            print(stats.round(2))
        else:
            print(f"\n📋 Analyse Catégorielle (Calculée) : {col_name}")
            freq = df_data[col_name].value_counts(dropna=False)
            freq_pct = (freq / total_global) * 100
            resume = pd.DataFrame({'Effectif': freq, 'Pourcentage (%)': freq_pct.round(2)})
            resume.index = ['NaN (non concernés)' if pd.isna(x) else str(x) for x in resume.index]
            print(resume.head(10).to_string())

# ==========================================
# 4. LANCEMENT DU MOTEUR
# ==========================================
df_survey_clean = preparer_regles_survey(df_survey)
executer_eda_univariee(df_data, df_survey_clean)

✅ Données chargées : 50000 lignes et 13 colonnes.

🔍 Analyse : enqueteur (Type XLSForm : select_one enqueteur)
--------------------------------------------------
📋 Tableau de fréquences (Total catégories : 25) :
              Effectif  Pourcentage (%)
enqueteur_2       2088             4.18
enqueteur_12      2070             4.14
enqueteur_5       2053             4.11
enqueteur_10      2049             4.10
enqueteur_22      2044             4.09
enqueteur_7       2039             4.08
enqueteur_16      2038             4.08
enqueteur_11      2029             4.06
enqueteur_6       2018             4.04
enqueteur_19      2010             4.02
enqueteur_21      2010             4.02
enqueteur_4       2007             4.01
enqueteur_18      2006             4.01
enqueteur_14      1994             3.99
enqueteur_17      1993             3.99
enqueteur_15      1990             3.98
enqueteur_9       1987             3.97
enqueteur_3       1973             3.95
enqueteur_13      1970      